# Notebook 4 — Patch Map Scoring: With vs. Without Map

For each of 10 val images and each target layer `L`, compute a per-patch score **with** and **without** the trained contrastive patch map:

| Mode | Input to CLS lens |
|------|-------------------|
| **Without map** | raw patch token `h[i,j]` ∈ ℝ^d |
| **With map** | `y[i,j] = map(h[i,j])` ∈ ℝ^d |

**Score** = `softmax(lens(input))[y_hat]` ∈ [0, 1], where `y_hat` is the model's top-1 prediction.

For each sample the figure shows:
- **Row 0**: original image │ *no-map* overlay per layer
- **Row 1**: prediction label │ *with-map* overlay per layer

In [ ]:
%matplotlib inline

import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.datasets as datasets

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from tuned_lens.model import VisionModelWrapper
from tuned_lens.config import ModelConfig
from tuned_lens.patch_map import FullPatchMap, LowRankPatchMap, PatchMapBank
from attribute_lens.scorer import load_lens_checkpoint, discover_lens_files

print('Imports OK')

In [ ]:
# ── CONFIGURATION — edit paths to match your environment ──────────────────────
IMAGENET_VAL_DIR = '/opt/models/datasets/imagenet/extracted/val'
CLS_LENS_DIR     = '../outputs/affine_kld/best_lenses'
PATCH_MAP_DIR    = '../outputs/patch_map_lowrank/best_maps'
MODEL_NAME       = 'vit_large_patch14_clip_224.openai_ft_in1k'

TARGET_LAYERS    = [0, 6, 12, 18, 23]   # will be filtered to available checkpoints
N_IMAGES         = 10
SEED             = 42
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Device       : {DEVICE}')
print(f'CLS lens dir : {CLS_LENS_DIR}')
print(f'Patch map dir: {PATCH_MAP_DIR}')

## SC 1 — Verify paths and discover available checkpoints

In [ ]:
assert Path(IMAGENET_VAL_DIR).exists(), f'Val dir not found: {IMAGENET_VAL_DIR}'
assert Path(CLS_LENS_DIR).exists(),     f'CLS lens dir not found: {CLS_LENS_DIR}'
assert Path(PATCH_MAP_DIR).exists(),    f'Patch map dir not found: {PATCH_MAP_DIR}'

available_lenses = discover_lens_files(CLS_LENS_DIR)
available_maps   = discover_lens_files(PATCH_MAP_DIR)   # same layer_N.pt convention

print(f'CLS lenses  : {sorted(available_lenses.keys())}')
print(f'Patch maps  : {sorted(available_maps.keys())}')

# Keep layers that have BOTH a lens and a map checkpoint
TARGET_LAYERS = sorted(
    l for l in TARGET_LAYERS
    if l in available_lenses and l in available_maps
)
assert TARGET_LAYERS, (
    'No layers have both a CLS lens and a patch map checkpoint.\n'
    f'  Lenses: {sorted(available_lenses.keys())}\n'
    f'  Maps  : {sorted(available_maps.keys())}'
)
print(f'Using layers: {TARGET_LAYERS}')
print('SC 1 ✓')

## Load model

In [ ]:
model_cfg = ModelConfig(
    model_name=MODEL_NAME,
    pretrained=True,
    target_layers=TARGET_LAYERS,
    freeze_model=True,
    patch_mode=False,  # extract_cls_and_patches handles both in one pass
)
wrapper = VisionModelWrapper(model_cfg, device=DEVICE)

H, W = wrapper.patch_grid_size
PATCH_SIZE = 224 // H   # e.g. 14 for ViT-L/14

print(f'Model      : {MODEL_NAME}')
print(f'd_model    : {wrapper.d_model}')
print(f'num_classes: {wrapper.num_classes}')
print(f'patch_grid : {H}x{W}  patch_size={PATCH_SIZE}px')
print(f'target_lyrs: {wrapper.target_layers}')

## SC 2 — Verify model properties

In [ ]:
assert wrapper.d_model > 0
assert wrapper.num_classes == 1000, f'Expected 1000 classes, got {wrapper.num_classes}'
assert wrapper.target_layers == TARGET_LAYERS, 'Target-layer mismatch'
assert hasattr(wrapper, 'extract_cls_and_patches'), 'extract_cls_and_patches not found on wrapper'
print(f'SC 2 ✓  d_model={wrapper.d_model}  num_classes={wrapper.num_classes}')

## Load ImageNet val dataset and sample 10 images

In [ ]:
transform   = wrapper.get_transform()
dataset     = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=transform)
raw_dataset = datasets.ImageFolder(IMAGENET_VAL_DIR)   # PIL images for visualization
class_names = dataset.classes

rng            = random.Random(SEED)
sample_indices = rng.sample(range(len(dataset)), N_IMAGES)

print(f'Val images : {len(dataset):,}  ({len(class_names)} classes)')
print(f'Sampled idx: {sample_indices}')

## SC 3 — Display the 10 sampled images

In [ ]:
img_chk, lbl_chk = dataset[sample_indices[0]]
assert img_chk.shape == (3, 224, 224), f'Expected (3,224,224), got {img_chk.shape}'
assert 0 <= lbl_chk < len(class_names)

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for ax, idx in zip(axes.flat, sample_indices):
    pil_img, lbl = raw_dataset[idx]
    ax.imshow(pil_img.resize((224, 224)))
    ax.set_title(f'{class_names[lbl][:18]}\nidx={idx}', fontsize=8)
    ax.axis('off')
plt.suptitle('10 Sampled ImageNet Val Images', fontsize=13)
plt.tight_layout()
plt.show()
print('SC 3 ✓')

## Extract CLS tokens and patch tokens for all sampled images

`extract_cls_and_patches` runs a single forward pass capturing both CLS (position 0)
and patch grid (positions 1:) at every target layer.

In [ ]:
all_cls_states   = {}   # {img_idx: {layer_idx: Tensor[d_model]}}
all_patch_states = {}   # {img_idx: {layer_idx: Tensor[H, W, d_model]}}
all_logits       = {}   # {img_idx: Tensor[num_classes]}
all_y_hats       = {}   # {img_idx: int}

for n, idx in enumerate(sample_indices):
    img_t, _ = dataset[idx]
    batch = img_t.unsqueeze(0).to(DEVICE)   # [1, 3, 224, 224]

    cls_dict, patch_dict, logits = wrapper.extract_cls_and_patches(batch)

    all_cls_states[idx]   = {k: v[0].cpu() for k, v in cls_dict.items()}     # drop batch dim
    all_patch_states[idx] = {k: v[0].cpu() for k, v in patch_dict.items()}   # [H, W, d]
    all_logits[idx]       = logits[0].cpu()
    all_y_hats[idx]       = int(logits[0].argmax())

    pred = class_names[all_y_hats[idx]]
    print(f'  [{n+1:2d}/{N_IMAGES}] idx={idx:6d}  y_hat={all_y_hats[idx]:4d}  ({pred})')

print('Extraction complete.')

## SC 4 — Verify extraction shapes

In [ ]:
idx0 = sample_indices[0]
for layer_idx in TARGET_LAYERS:
    cls_t   = all_cls_states[idx0][layer_idx]
    patch_t = all_patch_states[idx0][layer_idx]
    assert cls_t.shape   == (wrapper.d_model,),          f'CLS shape {cls_t.shape}'
    assert patch_t.shape == (H, W, wrapper.d_model),     f'Patch shape {patch_t.shape}'
    print(
        f'  Layer {layer_idx:2d}: CLS {tuple(cls_t.shape)}  '
        f'patches {tuple(patch_t.shape)}  ✓'
    )
print('SC 4 ✓')

## Load CLS lenses

In [ ]:
lenses = {}   # {layer_idx: BaseLens}
for layer_idx in TARGET_LAYERS:
    lenses[layer_idx] = load_lens_checkpoint(available_lenses[layer_idx], device=DEVICE)
    print(f'  Layer {layer_idx:2d}: {type(lenses[layer_idx]).__name__}')
print('CLS lenses loaded.')

## Load patch maps

Auto-detect `FullPatchMap` vs `LowRankPatchMap` from the state dict keys:
- `linear.weight` → `FullPatchMap`
- `down.weight`   → `LowRankPatchMap`

In [ ]:
def load_patch_map_checkpoint(path: str, device: str = 'cpu'):
    """Load a FullPatchMap or LowRankPatchMap from a .pt checkpoint."""
    payload = torch.load(path, map_location='cpu', weights_only=False)
    sd = payload['state_dict']

    if 'linear.weight' in sd:
        d_model = sd['linear.weight'].shape[0]   # square: [d, d]
        m = FullPatchMap(d_model)
    elif 'down.weight' in sd:
        d_model = sd['down.weight'].shape[1]     # [rank, d_model]
        rank    = sd['down.weight'].shape[0]
        m = LowRankPatchMap(d_model, rank)
    else:
        raise ValueError(f'Unrecognised patch map state dict keys: {list(sd.keys())}')

    m.load_state_dict(sd)
    m.to(device)
    m.eval()
    return m


patch_maps = {}   # {layer_idx: BasePatchMap}
for layer_idx in TARGET_LAYERS:
    patch_maps[layer_idx] = load_patch_map_checkpoint(available_maps[layer_idx], device=DEVICE)
    m = patch_maps[layer_idx]
    if isinstance(m, LowRankPatchMap):
        desc = f'LowRankPatchMap(rank={m.rank})'
    else:
        desc = f'FullPatchMap(d={m.d_model})'
    print(f'  Layer {layer_idx:2d}: {desc}')
print('Patch maps loaded.')

## SC 5 — Verify lens and map dimensions match d_model

In [ ]:
d = wrapper.d_model
C = wrapper.num_classes

for layer_idx in TARGET_LAYERS:
    lens = lenses[layer_idx]
    pm   = patch_maps[layer_idx]

    # Lens: input must be d_model, output must be num_classes
    if hasattr(lens, 'linear'):
        lens_in, lens_out = lens.linear.weight.shape[1], lens.linear.weight.shape[0]
    else:
        lins = [m for m in lens.net.modules() if hasattr(m, 'weight') and m.weight.dim() == 2]
        lens_in, lens_out = lins[0].weight.shape[1], lins[-1].weight.shape[0]

    # Map: input and output must both be d_model
    if isinstance(pm, FullPatchMap):
        map_in = map_out = pm.linear.weight.shape[1]
    else:
        map_in  = pm.down.weight.shape[1]   # d_model
        map_out = pm.up.weight.shape[0]     # d_model

    assert lens_in  == d, f'Layer {layer_idx}: lens input {lens_in} != d_model {d}'
    assert lens_out == C, f'Layer {layer_idx}: lens output {lens_out} != num_classes {C}'
    assert map_in   == d, f'Layer {layer_idx}: map input {map_in} != d_model {d}'
    assert map_out  == d, f'Layer {layer_idx}: map output {map_out} != d_model {d}'

    print(f'  Layer {layer_idx:2d}: lens [{lens_in}->{lens_out}]  map [{map_in}->{map_out}]  ✓')

print('SC 5 ✓')

## Compute score maps — with and without the patch map

For every patch `(i,j)` at every target layer:

```
score_no_map[i,j]   = softmax( lens(h[i,j]) )[y_hat]
score_with_map[i,j] = softmax( lens(map(h[i,j])) )[y_hat]
```

In [ ]:
# scores_no_map[img_idx][layer_idx]   = Tensor[H, W]
# scores_with_map[img_idx][layer_idx] = Tensor[H, W]
scores_no_map   = {}
scores_with_map = {}

for img_idx in sample_indices:
    scores_no_map[img_idx]   = {}
    scores_with_map[img_idx] = {}
    y_hat = all_y_hats[img_idx]

    for layer_idx in TARGET_LAYERS:
        patches = all_patch_states[img_idx][layer_idx].to(DEVICE)  # [H, W, d]
        flat    = patches.reshape(H * W, wrapper.d_model)           # [H*W, d]

        with torch.no_grad():
            # Without map: raw patch token -> lens -> softmax
            logits_no   = lenses[layer_idx](flat)                   # [H*W, C]
            probs_no    = F.softmax(logits_no, dim=-1)[:, y_hat].cpu()

            # With map: map(patch token) -> lens -> softmax
            mapped      = patch_maps[layer_idx](flat)               # [H*W, d]
            logits_with = lenses[layer_idx](mapped)                 # [H*W, C]
            probs_with  = F.softmax(logits_with, dim=-1)[:, y_hat].cpu()

        scores_no_map[img_idx][layer_idx]   = probs_no.reshape(H, W)
        scores_with_map[img_idx][layer_idx] = probs_with.reshape(H, W)

print('Scoring complete.')

## SC 6 — Verify score map properties

In [ ]:
idx0 = sample_indices[0]
for layer_idx in TARGET_LAYERS:
    sm_no   = scores_no_map[idx0][layer_idx]
    sm_with = scores_with_map[idx0][layer_idx]

    for label, sm in [('no_map', sm_no), ('with_map', sm_with)]:
        assert sm.shape == (H, W),           f'{label} layer {layer_idx}: shape {sm.shape}'
        assert not torch.isnan(sm).any(),    f'{label} layer {layer_idx}: unexpected NaN'
        assert (sm >= 0).all() and (sm <= 1).all(), \
            f'{label} layer {layer_idx}: scores out of [0,1]'

    delta_mean = (sm_with - sm_no).mean().item()
    print(
        f'  Layer {layer_idx:2d}: '
        f'no_map=[{sm_no.min():.4f},{sm_no.max():.4f}] mean={sm_no.mean():.4f}  '
        f'with_map=[{sm_with.min():.4f},{sm_with.max():.4f}] mean={sm_with.mean():.4f}  '
        f'Δmean={delta_mean:+.4f}  ✓'
    )

print('SC 6 ✓')

## SC 7 — Manual spot-check for a single patch

Recompute scores for patch (8, 8) in the first image at the first target layer by hand
and confirm they match the stored maps.

In [ ]:
idx0   = sample_indices[0]
layer0 = TARGET_LAYERS[0]
y_hat0 = all_y_hats[idx0]
pi, pj = 8, 8

tok = all_patch_states[idx0][layer0][pi, pj].unsqueeze(0).to(DEVICE)  # [1, d]

with torch.no_grad():
    manual_no   = float(F.softmax(lenses[layer0](tok), dim=-1)[0, y_hat0])
    manual_with = float(F.softmax(lenses[layer0](patch_maps[layer0](tok)), dim=-1)[0, y_hat0])

stored_no   = float(scores_no_map[idx0][layer0][pi, pj])
stored_with = float(scores_with_map[idx0][layer0][pi, pj])

print(f'Patch ({pi},{pj})  layer {layer0}  y_hat={y_hat0}')
print(f'  no_map  : manual={manual_no:.6f}  stored={stored_no:.6f}')
print(f'  with_map: manual={manual_with:.6f}  stored={stored_with:.6f}')

assert abs(manual_no   - stored_no)   < 1e-5, 'no_map mismatch'
assert abs(manual_with - stored_with) < 1e-5, 'with_map mismatch'
print('SC 7 ✓')

## Visualize: image + no-map heatmap + with-map heatmap per layer

**Layout per figure (one per sample):**
- Row 0: original image │ *no-map* overlay — layer L₀, L₁, …
- Row 1: prediction text │ *with-map* overlay — layer L₀, L₁, …

Each overlay is the score map (bilinearly upsampled to 224×224) blended onto the image.

In [ ]:
def _make_overlay(img_np, score_map_hw):
    """Normalise score_map, upsample to 224x224, return RGBA overlay array."""
    sm = score_map_hw.numpy()
    sm_norm = (sm - sm.min()) / (sm.max() - sm.min() + 1e-8)
    hm_u8   = (sm_norm * 255).astype(np.uint8)
    hm_224  = np.array(Image.fromarray(hm_u8).resize((224, 224), Image.BILINEAR))
    return hm_224


for img_idx in sample_indices:
    y_hat      = all_y_hats[img_idx]
    pil_img, _ = raw_dataset[img_idx]
    img_np     = np.array(pil_img.resize((224, 224)))
    pred_label = class_names[y_hat]

    n_cols = 1 + len(TARGET_LAYERS)
    fig, axes = plt.subplots(2, n_cols, figsize=(3.5 * n_cols, 7.5))

    # ── Column 0 ──────────────────────────────────────────────────────────
    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title(f'Original\nidx={img_idx}', fontsize=8)
    axes[0, 0].axis('off')

    axes[1, 0].text(
        0.5, 0.5, f'Pred:\n{pred_label[:22]}\n(class {y_hat})',
        ha='center', va='center', fontsize=8.5,
        transform=axes[1, 0].transAxes,
    )
    axes[1, 0].axis('off')

    # ── Columns 1..N: one per target layer ────────────────────────────────
    for col, layer_idx in enumerate(TARGET_LAYERS, start=1):
        sm_no   = scores_no_map[img_idx][layer_idx]
        sm_with = scores_with_map[img_idx][layer_idx]

        hm_no   = _make_overlay(img_np, sm_no)
        hm_with = _make_overlay(img_np, sm_with)

        # Row 0: no-map
        axes[0, col].imshow(img_np)
        axes[0, col].imshow(hm_no, cmap='hot', alpha=0.55, vmin=0, vmax=255)
        axes[0, col].set_title(
            f'L{layer_idx}  no map\n'
            f'[{sm_no.min():.3f}, {sm_no.max():.3f}]',
            fontsize=7.5,
        )
        axes[0, col].axis('off')

        # Row 1: with-map
        axes[1, col].imshow(img_np)
        axes[1, col].imshow(hm_with, cmap='hot', alpha=0.55, vmin=0, vmax=255)
        axes[1, col].set_title(
            f'L{layer_idx}  with map\n'
            f'[{sm_with.min():.3f}, {sm_with.max():.3f}]',
            fontsize=7.5,
        )
        axes[1, col].axis('off')

    plt.suptitle(
        f'idx={img_idx}  Pred: {pred_label}  —  '
        'Row 0: no map   Row 1: with map',
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

## Bonus: score difference map (with_map − no_map)

Positive (red) = the map *increased* the class score for that patch.
Negative (blue) = the map *decreased* it (patch likely suppressed / background).

In [ ]:
for img_idx in sample_indices:
    y_hat      = all_y_hats[img_idx]
    pil_img, _ = raw_dataset[img_idx]
    img_np     = np.array(pil_img.resize((224, 224)))
    pred_label = class_names[y_hat]

    n_cols = 1 + len(TARGET_LAYERS)
    fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 4))

    axes[0].imshow(img_np)
    axes[0].set_title(f'Original\n{pred_label[:20]}', fontsize=8)
    axes[0].axis('off')

    for col, layer_idx in enumerate(TARGET_LAYERS, start=1):
        diff = (scores_with_map[img_idx][layer_idx] - scores_no_map[img_idx][layer_idx]).numpy()
        vabs = max(abs(diff.min()), abs(diff.max()), 1e-8)

        # Upsample to 224x224
        diff_224 = np.array(
            Image.fromarray(diff.astype(np.float32)).resize((224, 224), Image.BILINEAR)
        )

        axes[col].imshow(img_np, alpha=0.35)
        im = axes[col].imshow(diff_224, cmap='RdBu_r', vmin=-vabs, vmax=vabs, alpha=0.75)
        plt.colorbar(im, ax=axes[col], fraction=0.046, pad=0.04)
        axes[col].set_title(
            f'L{layer_idx}  Δscore\nwith−no (mean={diff.mean():+.4f})',
            fontsize=7.5,
        )
        axes[col].axis('off')

    plt.suptitle(
        f'idx={img_idx}  Pred: {pred_label}  —  Δscore = with_map − no_map',
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()